# Model Optimization Workshop: Introduction and Setup

Welcome to the Model Optimization Workshop! In this workshop, you'll learn how to optimize machine learning models to improve performance and reduce costs. This first notebook will guide you through setting up your environment and downloading the necessary models.

# Part 1: Environment Setup

## 1. Install Required Packages

We'll install the basic packages needed for this notebook. Each subsequent notebook will install its specific dependencies as needed.

In [ ]:
# Install only the packages needed for this notebook
!pip install -q "transformers==4.26.0" "boto3>=1.26.0" "sagemaker>=2.130.0" "pandas>=1.5.0" "numpy>=1.23.0" "matplotlib>=3.6.0" "torch==1.13.1" "huggingface_hub>=0.14.1"

## 2. Verify Environment

In [ ]:
# Import standard libraries
import os
import sys
import time
import json
import logging

# Data processing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Machine learning
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# AWS
import boto3
import sagemaker

# Print versions of key packages
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Boto3 version: {boto3.__version__}")
print(f"SageMaker version: {sagemaker.__version__}")

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")

## 3. Set Up SageMaker Session

In [ ]:
# Set up SageMaker session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.session.Session().region_name
bucket = sagemaker_session.default_bucket()
prefix = "model-optimization-workshop"

print(f"SageMaker session established in region: {region}")
print(f"Using S3 bucket: {bucket}")
print(f"Using S3 prefix: {prefix}")

## 4. Create Workshop Configuration

Let's create a configuration file that will be used across all notebooks to ensure consistency.

In [ ]:
# Create workshop configuration
workshop_config = {
    "base_model": "distilbert-base-uncased-finetuned-sst-2-english",
    "task": "sequence-classification",
    "s3_bucket": bucket,
    "s3_prefix": prefix,
    "region": region,
    "role": role,
    "device_type": "GPU" if torch.cuda.is_available() else "CPU",
    "created_at": time.strftime("%Y-%m-%d-%H-%M-%S")
}

# Save configuration to file
with open("workshop_config.json", "w") as f:
    json.dump(workshop_config, f, indent=2)

print("Workshop configuration saved to workshop_config.json")

## 5. Download Base Model

We'll download the base model that will be used throughout the workshop.

In [ ]:
# Download base model
model_name = workshop_config["base_model"]
task = workshop_config["task"]

print(f"Downloading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
print(f"Model downloaded successfully")

# Get model size
def get_model_size(model):
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    size_mb = (param_size + buffer_size) / 1024**2
    return size_mb

model_size_mb = get_model_size(model)
print(f"Model size: {model_size_mb:.2f} MB")

## 6. Test Model Inference

Let's test the model with a simple inference example.

In [ ]:
# Test model inference
sample_text = "This workshop is really helpful and informative!"

# Prepare inputs
def prepare_inputs(task, tokenizer, sample_input):
    if task == "sequence-classification":
        inputs = tokenizer(sample_input, return_tensors="pt")
    elif task == "token-classification":
        inputs = tokenizer(sample_input, return_tensors="pt")
    elif task == "question-answering":
        inputs = tokenizer(
            sample_input["question"],
            sample_input["context"],
            return_tensors="pt"
        )
    elif task == "masked-lm":
        inputs = tokenizer(sample_input, return_tensors="pt")
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    return inputs

inputs = prepare_inputs(task, tokenizer, sample_text)

# Move model to appropriate device
model = model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Run inference
with torch.no_grad():
    outputs = model(**inputs)

# Get prediction
logits = outputs.logits
predicted_class = torch.argmax(logits, dim=1).item()
print(f"Input text: '{sample_text}'")
print(f"Predicted class: {predicted_class} ({'positive' if predicted_class == 1 else 'negative'})")

# Measure inference time
def measure_inference_time(model, inputs, num_runs=100, warmup_runs=10):
    # Warmup
    for _ in range(warmup_runs):
        _ = model(**inputs)
    
    # Measure inference time
    start_time = time.time()
    for _ in range(num_runs):
        _ = model(**inputs)
    end_time = time.time()
    
    avg_time = (end_time - start_time) / num_runs
    return avg_time * 1000  # Convert to milliseconds

inference_time = measure_inference_time(model, inputs)
print(f"Average inference time: {inference_time:.2f} ms")

## 7. Save Model Information

Let's save the base model information for comparison with optimized models later.

In [ ]:
# Save model information
model_info = {
    "base_model": {
        "name": model_name,
        "task": task,
        "size_mb": model_size_mb,
        "inference_time_ms": inference_time
    }
}

# Save to file
with open("model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

print("Base model information saved to model_info.json")

# Next Steps

You've successfully set up your environment and downloaded the base model. In the next notebook, we'll explore quantization techniques to reduce the model size while maintaining performance.